# Who Pays What? (starter)

Read `README.md` first. This notebook loads the data, trains a plain baseline on the numeric columns, and scores it. Your job: beat it.

In [1]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score

TARGET = "ukt_bracket"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)
train.head()

(24304, 28) (6076, 27)


,id,family_income,num_dependents,parent_occupation,parent_education,asset_ownership,home_location,family_monthly_spending,gov_aid_program,scholarship_history,...,roof_material,wall_material,floor_material,utilities_monthly_spending,total_debt,monthly_debt_installment,study_program,city,admission_route,ukt_bracket
0,105983,"IDR 1,000,000 to IDR 2,000,000",0.0,Laborer,SLTP,Own,Reg. Temanggung,600000.0,KIS/JKS/BPJS,KIPK,...,NaN,Stone brick,Ceramic,50000.0,0.0,0.0,Development Economics,Reg. Temanggung,SNBP,0
1,123651,"more than IDR 5,000,000",1.0,Laborer,SLTA,NaN,Reg. Demak,8000000.0,KIS/JKS/BPJS,Not received,...,Tiles earth clay,Stone brick,Ceramic,75321.0,20000000.0,2000000.0,Fine Arts Education,Reg. Demak,SNBP,6
2,95611,"less than IDR 1,000,000,-",1.0,Not working,SD,Own,Regency Rembang,1000000.0,KIS/JKS/BPJS,kipk,...,TILE,CEMENT STONE BRICK,CEMENT PLASTR,30000.0,100000.0,100000.0,Sports Coaching Education,Regency Rembang,SNBT,0
3,95829,"IDR 4,000,000 to IDR 5,000,000",1.0,Laborer,SMA,Own,Regency Jepara,1500000.0,KIS/JKS/BPJS,No,...,Earth Clay,Stone Brick,Ceramic,150000.0,300000.0,0.0,Primary School Teacher Education,Regency Jepara,SNBT,4
4,108368,"IDR 3,000,000 to IDR 4,000,000",1.0,Farmer,SMA,Own,Reg. Lamongan,1300000.0,KIS/JKS/BPJS,-,...,Tiles from earth clay,stone brick,ceramic,280.0,26130500.0,0.0,Architecture,Reg. Lamongan,SNBT,5


## A quick look at the messy text

In [2]:
for col in ["roof_material", "wall_material", "floor_material"]:
    print(col, "->", train[col].nunique(), "different answers")
    print(train[col].value_counts().head(8).to_dict(), "\n")

roof_material -> 1339 different answers
{'Tile': 3949, 'Tiles': 3557, 'Tile earth clay': 1742, 'tiles': 1580, 'Asbestos': 1229, 'tile': 1172, 'Tiles earth clay': 903, 'Zinc': 891} 

wall_material -> 2775 different answers
{'Stone brick': 3318, 'stone brick': 2211, 'Stone Brick': 1594, 'Brick': 1198, 'Cement': 1192, 'Cinderblock': 844, 'STONE BRICK': 814, 'Wood': 689} 

floor_material -> 1658 different answers
{'Ceramic': 9692, 'ceramic': 4345, 'CERAMIC': 1656, 'Floortile': 543, 'Cement': 485, 'Plaster': 454, 'Earth': 398, 'Ceramc': 319} 



## Scoring helper
Use it only at the end. For decisions, use cross-validation on `train`.

In [3]:
def score(pred):
    """pred: array of brackets (0-7), same order as test.csv"""
    labels = pd.read_csv("data/test_labels.csv").set_index("id").loc[test["id"], TARGET].values
    f1 = f1_score(labels, pred, average="macro")
    print(f"test macro-F1 = {f1:.4f}   accuracy = {accuracy_score(labels, pred):.4f}")
    return f1

## Baseline: XGBoost on the numeric columns

In [4]:
from xgboost import XGBClassifier

numeric = [c for c in test.columns if test[c].dtype != object and c != "id"]
model = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=42)
model.fit(train[numeric], train[TARGET])
pred = model.predict(test[numeric])

## Your predictions
One row per test form, columns `id, ukt_bracket`.

In [5]:
submission = pd.DataFrame({"id": test["id"], TARGET: pred})
submission.head()

,id,ukt_bracket
0,108679,4
1,111114,4
2,94611,4
3,127260,4
4,88864,0


In [6]:
score(submission[TARGET])

test macro-F1 = 0.2533   accuracy = 0.3221


0.2533303499073322

## Ideas to try
- Many text columns hide a small number of real answers behind many spellings. How could you group spellings that are *almost* the same?
- Some categories have an order (income ranges, education levels).
- Look at bracket 0 in `train`. What do those students have in common?
- Macro-F1 punishes ignoring a rare bracket.